In [43]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn

In [44]:
train_df= pd.read_csv('train_data.csv')
test_df= pd.read_csv('test_data.csv')

X_train= train_df['sequence'].to_list()

y_train = [
    list(map(float, s.split()))
    for s in train_df['target_profile']
]

X_test= test_df['sequence'].to_list()

vocab= {'A': 1, 'U':2, 'C':3, 'G':4}

In [45]:
from torch.utils.data import Dataset, DataLoader

class RNA_dataset(Dataset):
    def __init__(self, sequences, targets= None):
        self.sequences= sequences
        self.targets= targets
        
    def __len__(self):
        return len(self.sequences)
    
    def __getitem__(self, idx):
        returned_x= torch.LongTensor([vocab[letter] for letter in self.sequences[idx]])
        if not self.targets:
            return returned_x, len(self.sequences[idx])
        returned_y= torch.FloatTensor(self.targets[idx])
        return returned_x, returned_y

In [46]:
from torch.nn.utils.rnn import pad_sequence

train_dataset= RNA_dataset(X_train, y_train)
test_dataset= RNA_dataset(X_test)

def collate_fn_train(batch):
    x, y = zip(*batch)
    x_padded = pad_sequence(x, batch_first=True, padding_value=0) 
    y_padded = pad_sequence(y, batch_first=True, padding_value=0) 
    return x_padded, y_padded

def collate_fn_test(batch):
    x, lengths= zip(*batch)
    x_padded = pad_sequence(x, batch_first=True, padding_value=0) 
    return x_padded, lengths

train_dataloader= DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn= collate_fn_train)
test_dataloader= DataLoader(test_dataset, batch_size=32, shuffle=False, collate_fn= collate_fn_test)

In [47]:
class lstm(nn.Module):

    def __init__(self, vocab_size=5, embed_dim=8, hidden_dim=32):
        super(lstm, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers= 3, batch_first=True, bidirectional=True, dropout=0.3)
        self.fc = nn.Linear(hidden_dim*2, 1)
    
    def forward(self,x):
        x = self.embedding(x)
        lstm_out, _ = self.lstm(x)
        prediction = self.fc(lstm_out)
        return prediction.squeeze(-1)

In [48]:
epochs= 20
lr= 1e-3

model= lstm()
criterion= nn.MSELoss()
optimizer= torch.optim.AdamW(model.parameters(), lr= lr)

for epoch in range(epochs):
    model.train()
    running_loss= 0.0
    for x,y in train_dataloader:
        optimizer.zero_grad()
        output= model(x)
        c= (x!= 0)
        loss= criterion(output[c], y[c])
        loss.backward()
        optimizer.step()
        running_loss+= loss
        
    if (epoch+1)% 5== 0:
        print(f'Epoch:{epoch+ 1}, loss:{running_loss/len(train_dataloader):.6f}')

Epoch:5, loss:0.006729
Epoch:10, loss:0.003754
Epoch:15, loss:0.002984
Epoch:20, loss:0.002815


In [49]:
model.eval()
preds = []

with torch.no_grad():
    for x, lengths in test_dataloader:
        output = model(x)
        for pred, l in zip(output, lengths):
            preds.append(pred[:l].tolist())

preds_str = [' '.join(map(str,i)) for i in preds]

In [50]:
answer = pd.DataFrame({
    'subtaskID':1,
    'datapointID': test_df['id'],
    'answer': preds_str
})
answer.to_csv("submisison.csv", index= False)